# 05: Basic Sentiment Analysis - Is This Review Positive?

## Your First NLP Project!

Now we'll apply everything we've learned to a real NLP problem: **sentiment analysis**.

Given a text review, can we predict if it's positive or negative?

### The Web Dev Analogy

Think of this like spam detection:
- Input: Text content
- Output: Category (spam/not spam, positive/negative)
- Method: Look for patterns in words

## What You'll Learn
- [ ] Build a complete text classification pipeline from scratch
- [ ] Explain how bag-of-words converts text to numerical features
- [ ] Interpret model weights to understand what the model learned

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 2**: Bag-of-words representation | BoW becomes our feature extraction — each review becomes a word-count vector |
| **Lesson 4**: Logistic regression | Our classifier! Sigmoid turns word-count features into sentiment probabilities |

> Together, BoW + logistic regression form your first complete NLP system!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to analyze sentiment! 💭")

## 1. Our Dataset: Movie Reviews

Let's create a simple movie review dataset:

In [ ]:
# Simple movie review dataset
reviews = [
    # Positive reviews (label = 1)
    ("This movie was absolutely amazing! I loved every minute.", 1),
    ("Fantastic film with great acting. Highly recommended!", 1),
    ("A wonderful story that moved me to tears. Beautiful!", 1),
    ("Best movie I've seen this year. Outstanding performance.", 1),
    ("Incredible cinematography and a touching story. Loved it!", 1),
    ("The acting was superb and the plot was engaging.", 1),
    ("A masterpiece! This film exceeded all expectations.", 1),
    ("Heartwarming and funny. A perfect feel-good movie.", 1),
    ("Brilliant direction and stellar performances throughout.", 1),
    ("This movie made me laugh and cry. Absolutely wonderful!", 1),
    ("An excellent film that keeps you engaged from start to finish.", 1),
    ("The best movie of the decade. A true cinematic achievement.", 1),
    
    # Negative reviews (label = 0)
    ("Terrible movie. Complete waste of time and money.", 0),
    ("Boring and predictable. I fell asleep halfway through.", 0),
    ("The worst film I've ever seen. Awful acting.", 0),
    ("Disappointing and dull. Not worth watching.", 0),
    ("A disaster of a movie. Poor writing and bad direction.", 0),
    ("I hated this film. It was painfully slow and boring.", 0),
    ("Waste of money. The plot made no sense at all.", 0),
    ("Horrible acting and a ridiculous storyline. Avoid!", 0),
    ("This movie was a complete disappointment. So bad.", 0),
    ("Unwatchable garbage. I want my two hours back.", 0),
    ("A terrible mess from start to finish. Just awful.", 0),
    ("The acting was wooden and the story was nonsensical.", 0),
]

# Shuffle the data
np.random.shuffle(reviews)

# Split into texts and labels
texts = [r[0] for r in reviews]
labels = np.array([r[1] for r in reviews])

print(f"Dataset: {len(texts)} reviews")
print(f"Positive: {sum(labels)} | Negative: {len(labels) - sum(labels)}")
print("\nSample reviews:")
for i in range(3):
    sentiment = "😊 Positive" if labels[i] == 1 else "😞 Negative"
    print(f"  {sentiment}: \"{texts[i][:50]}...\"")

## 2. Text Preprocessing

Before we can feed text to our model, we need to clean it up:

In [ ]:
def preprocess_text(text):
    """
    Clean and tokenize text:
    1. Convert to lowercase
    2. Remove punctuation
    3. Split into words
    """
    # Lowercase
    text = text.lower()
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Split into words
    words = text.split()
    
    return words

# Example
sample = "This movie was AMAZING! Best film of 2023."
print(f"Original: {sample}")
print(f"Processed: {preprocess_text(sample)}")

In [ ]:
# Preprocess all reviews
tokenized_reviews = [preprocess_text(text) for text in texts]

# Build vocabulary from all words
all_words = [word for review in tokenized_reviews for word in review]
word_counts = Counter(all_words)

print(f"Total words: {len(all_words)}")
print(f"Unique words: {len(word_counts)}")
print(f"\nMost common words:")
for word, count in word_counts.most_common(15):
    print(f"  '{word}': {count}")

## 3. Bag of Words: Text as Numbers

We'll use **Bag of Words** (BoW) to convert text to numbers:
- Each unique word is a feature
- Each document is represented by word counts

In [ ]:
class BagOfWords:
    """Simple Bag of Words vectorizer."""
    
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.vocabulary = {}
        self.word_to_idx = {}
    
    def fit(self, texts):
        """Build vocabulary from texts."""
        # Count all words
        word_counts = Counter()
        for words in texts:
            word_counts.update(words)
        
        # Keep words above min frequency
        vocab_words = [word for word, count in word_counts.items() 
                       if count >= self.min_freq]
        vocab_words = sorted(vocab_words)  # Sort for consistency
        
        self.vocabulary = vocab_words
        self.word_to_idx = {word: idx for idx, word in enumerate(vocab_words)}
        
        print(f"Vocabulary size: {len(self.vocabulary)}")
        return self
    
    def transform(self, texts):
        """Convert texts to BoW vectors."""
        vectors = np.zeros((len(texts), len(self.vocabulary)))
        
        for i, words in enumerate(texts):
            for word in words:
                if word in self.word_to_idx:
                    vectors[i, self.word_to_idx[word]] += 1
        
        return vectors
    
    def fit_transform(self, texts):
        """Fit and transform in one step."""
        self.fit(texts)
        return self.transform(texts)

# Create and fit vectorizer
vectorizer = BagOfWords(min_freq=2)  # Only words that appear 2+ times
X = vectorizer.fit_transform(tokenized_reviews)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Each review is now a vector of {X.shape[1]} features!")

In [ ]:

# --- See BoW in action on one review ---
i = 0
print(f"Review:    \"{texts[i][:70]}\"")
print(f"Sentiment: {'Positive ✓' if labels[i]==1 else 'Negative ✗'}")
print(f"Tokenized: {tokenized_reviews[i]}")
print()
print(f"As a BoW vector (vocabulary size: {X.shape[1]}):")
nonzero = [(vectorizer.vocabulary[j], int(X[i, j])) for j in range(X.shape[1]) if X[i, j] > 0]
for word, count in nonzero:
    print(f"  '{word}' → count = {count}")
zeros_count = X.shape[1] - len(nonzero)
print(f"  (+ {zeros_count} zeros for words not in this review)")
print(f"\n→ This review is now a vector of {X.shape[1]} numbers, ready for the model!")


In [ ]:
# Visualize a few reviews as vectors
fig, ax = plt.subplots(figsize=(14, 4))

# Show first 5 reviews
sample_X = X[:5]
im = ax.imshow(sample_X, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, label='Word Count')

# Labels
ax.set_yticks(range(5))
ax.set_yticklabels([f"Review {i+1} ({'Pos' if labels[i] else 'Neg'})" for i in range(5)])
ax.set_xlabel('Word Index in Vocabulary')
ax.set_title('Bag of Words: Each Review as a Vector of Word Counts')

plt.tight_layout()
plt.show()

## 4. Train/Test Split

In [ ]:
def train_test_split(X, y, test_size=0.25, random_state=None):
    """Split data into train and test sets."""
    if random_state:
        np.random.seed(random_state)
    
    n = len(X)
    n_test = int(n * test_size)
    
    indices = np.random.permutation(n)
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.25, random_state=42)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")


class LogisticRegression:
    """Logistic Regression classifier."""
    
    def __init__(self, learning_rate=0.1):
        self.lr = learning_rate        # Step size for gradient descent (hyperparameter — we choose this before training)
        self.weights = None             # One weight per vocabulary word. Starts as None, initialized in fit()
        self.bias = None                # Bias — one number that shifts the decision boundary
        self.history = {'loss': [], 'train_acc': [], 'test_acc': []}  # Training logs for plotting
    
    def sigmoid(self, z):
        # Squeezes any number to (0, 1) — converts a raw score into a probability
        # np.clip keeps z in [-500, 500] to avoid overflow in exp()
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def predict_proba(self, X):
        # Step 1: linear part — z = X·w + b (same as y = mx + b, but for many features)
        # X.shape = (num_reviews, vocab_size)
        # self.weights.shape = (vocab_size,)
        # z.shape = (num_reviews,) — one raw score per review
        z = np.dot(X, self.weights) + self.bias
        # Step 2: sigmoid converts z to a probability (0..1)
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        # Get probabilities, then apply threshold:
        # >= 0.5 → class 1 (positive), < 0.5 → class 0 (negative)
        # .astype(int) converts True/False to 1/0
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def compute_loss(self, y_true, y_pred):
        # Binary Cross-Entropy — measures how wrong our predictions are
        # epsilon prevents log(0) which would give -infinity
        epsilon = 1e-15
        # Clip predictions to [epsilon, 1-epsilon] — never exactly 0 or 1
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        # BCE: when y_true=1, penalize low y_pred; when y_true=0, penalize high y_pred
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X_train, y_train, X_test=None, y_test=None, n_iterations=1000, verbose=True):
        # n_samples = number of reviews, n_features = vocab size (= number of weights)
        n_samples, n_features = X_train.shape
        
        # Initialize all weights to zero — model starts knowing nothing
        # All predictions will be sigmoid(0) = 0.5 (total uncertainty)
        self.weights = np.zeros(n_features)  # one weight per word in vocabulary
        self.bias = 0
        
        # Main training loop — each iteration = one step down the loss "hill"
        for i in range(n_iterations):
            
            # === FORWARD PASS: what does the model predict right now? ===
            y_pred = self.predict_proba(X_train)  # probabilities for all reviews
            
            # === LOSS: how wrong is the model? ===
            loss = self.compute_loss(y_train, y_pred)  # one number — average error
            self.history['loss'].append(loss)           # save for plotting
            
            # === ACCURACY: fraction correct (for monitoring, not used in training) ===
            train_acc = np.mean(self.predict(X_train) == y_train)
            self.history['train_acc'].append(train_acc)
            
            # If test data provided, compute test accuracy too
            # (to watch for overfitting: train improves while test stalls)
            if X_test is not None:
                test_acc = np.mean(self.predict(X_test) == y_test)
                self.history['test_acc'].append(test_acc)
            
            # === GRADIENTS: which direction should weights move? ===
            # errors = "prediction minus reality" for each review
            # Positive error → model over-predicted → need to decrease weight
            # Negative error → model under-predicted → need to increase weight
            errors = y_pred - y_train  # shape: (n_samples,)
            
            # Weight gradient: "how responsible is each word for the errors?"
            # X_train.T @ errors: for each word, sum (word_count × error) across all reviews
            # If a word appears often in over-predicted reviews → decrease its weight
            # (1/n_samples) — average so gradient doesn't scale with dataset size
            d_weights = (1/n_samples) * np.dot(X_train.T, errors)  # shape: (n_features,)
            
            # Bias gradient: average error across all reviews
            # Bias isn't multiplied by any feature, so it's just the mean of errors
            d_bias = (1/n_samples) * np.sum(errors)  # one number
            
            # === UPDATE: step down the hill ===
            # Minus sign: gradient points UP (where loss grows), we want to go DOWN
            # self.lr controls step size — too large → overshoot, too small → slow
            self.weights -= self.lr * d_weights
            self.bias -= self.lr * d_bias
            
            # Print progress every 200 iterations
            if verbose and i % 200 == 0:
                test_str = f", Test Acc = {test_acc:.2%}" if X_test is not None else ""
                print(f"Iteration {i:4d}: Loss = {loss:.4f}, Train Acc = {train_acc:.2%}{test_str}")
        
        return self  # Return self so you can chain: model = LogisticRegression().fit(X, y)

# Train the model
print("Training sentiment classifier...\n")
model = LogisticRegression(learning_rate=0.1)
model.fit(X_train, y_train, X_test, y_test, n_iterations=1000)

print(f"\n✅ Final Train Accuracy: {model.history['train_acc'][-1]:.2%}")
print(f"✅ Final Test Accuracy: {model.history['test_acc'][-1]:.2%}")


## 5. Train Our Sentiment Classifier

In [ ]:
class LogisticRegression:
    """Logistic Regression classifier."""
    
    def __init__(self, learning_rate=0.1):
        self.lr = learning_rate        # Размер шага gradient descent (гиперпараметр, выбираем сами)
        self.weights = None             # Веса — по одному на каждое слово в словаре. Пока None, заполнятся в fit()
        self.bias = None                # Смещение — одно число, сдвигает границу решения
        self.history = {'loss': [], 'train_acc': [], 'test_acc': []}  # Логи обучения для графиков в cell-14
    
    def sigmoid(self, z):
        # Сжимаем любое число в диапазон (0, 1) — превращаем в вероятность
        # np.clip ограничивает z в [-500, 500] чтобы избежать переполнения в exp()
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def predict_proba(self, X):
        # Шаг 1: линейная часть — z = X @ w + b (то же y = mx + b, но для многих признаков)
        # X.shape = (кол-во review'ов, кол-во слов в словаре)
        # self.weights.shape = (кол-во слов в словаре,)
        # z.shape = (кол-во review'ов,) — одно число на каждый review
        z = np.dot(X, self.weights) + self.bias
        # Шаг 2: sigmoid превращает z в вероятность (0..1)
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        # Получаем вероятности, затем пороговое решение:
        # >= 0.5 → класс 1 (positive), < 0.5 → класс 0 (negative)
        # .astype(int) превращает True/False в 1/0
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def compute_loss(self, y_true, y_pred):
        # BCE (Binary Cross-Entropy) — измеряет насколько плохи наши предсказания
        # epsilon нужен чтобы не считать log(0), что дало бы -infinity
        epsilon = 1e-15
        # Ограничиваем предсказания в [epsilon, 1-epsilon] — никогда ровно 0 или 1
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        # Формула BCE: когда y_true=1, штрафуем за малые y_pred; когда y_true=0, штрафуем за большие
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X_train, y_train, X_test=None, y_test=None, n_iterations=1000, verbose=True):
        # n_samples = сколько review'ов, n_features = сколько слов в словаре (= кол-во весов)
        n_samples, n_features = X_train.shape
        
        # Инициализируем все веса нулями — модель пока "ничего не знает"
        # Все предсказания будут sigmoid(0) = 0.5 (полная неуверенность)
        self.weights = np.zeros(n_features)  # один вес на каждое слово в словаре
        self.bias = 0
        
        # Главный цикл обучения — каждая итерация = один шаг вниз по "холму" loss
        for i in range(n_iterations):
            
            # === FORWARD PASS: что модель думает сейчас? ===
            y_pred = self.predict_proba(X_train)  # вероятности для всех review'ов
            
            # === LOSS: насколько модель ошибается? ===
            loss = self.compute_loss(y_train, y_pred)  # одно число — средняя ошибка
            self.history['loss'].append(loss)           # сохраняем для графика
            
            # === ACCURACY: доля правильных ответов (для наблюдения, не для обучения) ===
            train_acc = np.mean(self.predict(X_train) == y_train)
            self.history['train_acc'].append(train_acc)
            
            # Если передали тестовые данные — считаем accuracy и на них
            # (чтобы следить за overfitting: train растёт, а test нет → модель зубрит)
            if X_test is not None:
                test_acc = np.mean(self.predict(X_test) == y_test)
                self.history['test_acc'].append(test_acc)
            
            # === ГРАДИЕНТЫ: в какую сторону двигать веса? ===
            # errors = разница "предсказание минус реальность" для каждого review'а
            # Положительная ошибка → модель завысила → нужно уменьшить
            # Отрицательная ошибка → модель занизила → нужно увеличить
            errors = y_pred - y_train  # shape: (n_samples,)
            
            # Градиент для весов: "насколько каждое слово виновато в ошибках"
            # X_train.T @ errors: для каждого слова суммируем (наличие слова × ошибка) по всем review'ам
            # Если слово часто встречается в review'ах с положительной ошибкой → его вес нужно уменьшить
            # (1/n_samples) — берём среднее, чтобы градиент не зависел от размера датасета
            d_weights = (1/n_samples) * np.dot(X_train.T, errors)  # shape: (n_features,)
            
            # Градиент для bias: средняя ошибка по всем review'ам
            # Bias не умножается на X, поэтому просто сумма ошибок
            d_bias = (1/n_samples) * np.sum(errors)  # одно число
            
            # === UPDATE: шагаем вниз по холму ===
            # Минус, потому что градиент указывает ВВЕРХ (куда loss растёт), а мы хотим ВНИЗ
            # self.lr контролирует размер шага
            self.weights -= self.lr * d_weights
            self.bias -= self.lr * d_bias
            
            # Печатаем прогресс каждые 200 итераций
            if verbose and i % 200 == 0:
                test_str = f", Test Acc = {test_acc:.2%}" if X_test is not None else ""
                print(f"Iteration {i:4d}: Loss = {loss:.4f}, Train Acc = {train_acc:.2%}{test_str}")
        
        return self  # возвращаем self чтобы можно было писать model = LogisticRegression().fit(X, y)

# Train the model
print("Training sentiment classifier...\n")
model = LogisticRegression(learning_rate=0.1)
model.fit(X_train, y_train, X_test, y_test, n_iterations=1000)

print(f"\n✅ Final Train Accuracy: {model.history['train_acc'][-1]:.2%}")
print(f"✅ Final Test Accuracy: {model.history['test_acc'][-1]:.2%}")

In [ ]:
# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(model.history['loss'])
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

# Accuracy
axes[1].plot(model.history['train_acc'], label='Train')
axes[1].plot(model.history['test_acc'], label='Test')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Over Time')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Interpreting the Model: What Words Matter?

One of the great things about logistic regression is **interpretability**. We can see which words the model thinks are positive or negative:

In [ ]:
# Get word importance from weights
word_weights = list(zip(vectorizer.vocabulary, model.weights))

# Sort by weight
word_weights.sort(key=lambda x: x[1])

# Most negative words (predict class 0)
print("🔴 Most NEGATIVE indicator words:")
for word, weight in word_weights[:10]:
    print(f"  '{word}': {weight:.3f}")

print("\n🟢 Most POSITIVE indicator words:")
for word, weight in word_weights[-10:][::-1]:
    print(f"  '{word}': {weight:.3f}")

In [ ]:
# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Negative words
neg_words = word_weights[:10]
axes[0].barh([w[0] for w in neg_words], [w[1] for w in neg_words], color='red', alpha=0.7)
axes[0].set_xlabel('Weight (more negative = stronger negative indicator)')
axes[0].set_title('Top 10 Negative Indicator Words')
axes[0].invert_yaxis()

# Positive words
pos_words = word_weights[-10:][::-1]
axes[1].barh([w[0] for w in pos_words], [w[1] for w in pos_words], color='green', alpha=0.7)
axes[1].set_xlabel('Weight (more positive = stronger positive indicator)')
axes[1].set_title('Top 10 Positive Indicator Words')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n🔍 The model learned which words suggest positive vs negative sentiment!")

## 7. Making Predictions on New Reviews

In [ ]:
def predict_sentiment(text, model, vectorizer):
    """Predict sentiment for a new review."""
    # Preprocess
    words = preprocess_text(text)
    
    # Vectorize
    vector = vectorizer.transform([words])
    
    # Predict
    prob = model.predict_proba(vector)[0]
    prediction = "Positive" if prob >= 0.5 else "Negative"
    confidence = prob if prob >= 0.5 else 1 - prob
    
    return prediction, prob, confidence

# Test on new reviews
new_reviews = [
    "This movie was absolutely incredible! A must-see masterpiece.",
    "Terrible film, complete waste of money. So boring.",
    "It was okay, nothing special but not bad either.",
    "I loved the acting but the story was disappointing.",
]

print("Predictions on new reviews:")
print("=" * 60)

for review in new_reviews:
    pred, prob, conf = predict_sentiment(review, model, vectorizer)
    emoji = "😊" if pred == "Positive" else "😞"
    print(f"\n{emoji} {pred} (confidence: {conf:.1%})")
    print(f"   \"{review[:50]}...\"" if len(review) > 50 else f"   \"{review}\"")

## 8. Limitations and Next Steps

Our simple model works, but has limitations:

1. **Ignores word order**: "not good" = "good not" (both have same counts)
2. **No word relationships**: "amazing" and "fantastic" are unrelated features
3. **Limited vocabulary**: Can't handle words it hasn't seen

We'll address these in upcoming modules with:
- **N-grams** (word pairs/triplets)
- **Word embeddings** (Module 5)
- **Sequence models** (Module 6)

## ⚠️ What Can Go Wrong: Overfitting on Small Data

With only 24 reviews, our model can easily **memorize** the training data
instead of learning generalizable patterns. Let's see what happens
when we train for too long.

In [ ]:
# --- Train for WAY too long and watch overfitting ---
print("Training for 5000 iterations (too many for 24 reviews!)")
print("=" * 55)

overfit_model = LogisticRegression(learning_rate=0.1)
overfit_model.fit(X_train, y_train, X_test, y_test, n_iterations=5000, verbose=False)

# Plot the divergence
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(overfit_model.history['train_acc'], label='Train Accuracy', color='blue')
ax.plot(overfit_model.history['test_acc'], label='Test Accuracy', color='red', linestyle='--')
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Train Accuracy ↑ but Test Accuracy Stalls')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

final_train = overfit_model.history['train_acc'][-1]
final_test = overfit_model.history['test_acc'][-1]
print(f"\nFinal Train Accuracy: {final_train:.2%}")
print(f"Final Test Accuracy:  {final_test:.2%}")
print(f"\n🔑 The gap tells the story:")
print(f"   Train ≈ 100% but Test stuck around {final_test:.0%}")
print(f"   → The model MEMORIZED {len(X_train)} training reviews,")
print(f"     but didn't learn generalizable patterns!")
print(f"\n💡 Rule of thumb: if train >> test accuracy, you're overfitting.")
print(f"   Fixes: more data, regularization, or early stopping.")

## 🌎 Now With Real Data: 20 Newsgroups

Our 24 hand-written reviews gave ~100% accuracy. Let's try a real dataset
from `sklearn.datasets` to see what realistic NLP performance looks like.

In [ ]:
# --- Real Data: 20 Newsgroups (positive vs negative topic classification) ---
from sklearn.datasets import fetch_20newsgroups

# Use 2 categories as a binary classification task
categories = ['rec.sport.baseball', 'sci.space']
newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

real_texts = newsgroups.data
real_labels = newsgroups.target  # 0 = baseball, 1 = space

print(f"Real dataset: {len(real_texts)} documents")
print(f"Classes: {newsgroups.target_names}")
print(f"Class distribution: {dict(zip(*np.unique(real_labels, return_counts=True)))}")

# Show a sample
print(f"\n--- Sample document (first 200 chars) ---")
print(f"'{real_texts[0][:200]}...'")
print(f"Label: {newsgroups.target_names[real_labels[0]]}")

In [ ]:

# --- Exercise 1: Build a BoW Vector Manually ---
# Without using vectorizer.transform(), build the BoW vector for this review from scratch.
# Steps:
#   1. Preprocess the text with preprocess_text()
#   2. Create a zero vector of length len(vectorizer.vocabulary)
#   3. For each word in the tokenized review, if it's in vectorizer.word_to_idx,
#      increment manual_vector at that index by 1

review = "This movie was absolutely wonderful"

# YOUR CODE HERE:
words = preprocess_text(review)
manual_vector = np.zeros(len(vectorizer.vocabulary))
# for word in words:
#     if word in vectorizer.word_to_idx:
#         manual_vector[...] += 1

# --- Check ---
expected = vectorizer.transform([words])[0]
assert np.any(manual_vector > 0), "Your vector is all zeros — did you fill in the words?"
assert np.allclose(manual_vector, expected), \
    "Vector doesn't match! Check that you're using vectorizer.word_to_idx to get positions."
print(f"Exercise 1 passed! ✓  ({int(np.sum(manual_vector > 0))} non-zero entries found)")

# --- Exercise 2: Most Positive Word ---
# Find the word with the highest positive weight in the trained model.
# Use vectorizer.vocabulary (list) and model.weights (array).

# YOUR CODE HERE:
top_word_idx = None  # index of the highest weight: use np.argmax(model.weights)
top_word = None      # the word at that index: vectorizer.vocabulary[top_word_idx]

# --- Check ---
assert top_word is not None, "Find the top positive word!"
assert top_word_idx == np.argmax(model.weights), "Use np.argmax on model.weights"
print(f"Exercise 2 passed! ✓  (Most positive word: '{top_word}', weight: {model.weights[top_word_idx]:.3f})")

# --- Quick Check: BoW Limitation ---
# What is the MAIN limitation of bag-of-words for sentiment analysis?
# a) It can't handle long reviews
# b) It requires too much memory
# c) It can't handle unknown words
# d) It ignores word order ("not good" = "good not")

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'd', "Think about 'not good' vs 'good not' — BoW gives identical vectors for both!"
print("Exercise 3 passed! ✓")

print("\n🎉 All exercises passed!")


## 📝 Check Your Understanding

1. Why do we preprocess text before feeding it to the model?
2. What does a positive weight for a word mean?
3. Why might "not good" be classified incorrectly?
4. How would you improve this classifier?

In [ ]:
# --- Exercise 1: Predict Sentiment ---
# Use the predict_sentiment function to classify this review.

# YOUR CODE HERE:
test_review = "This was absolutely terrible and boring"
pred, prob, conf = predict_sentiment(test_review, model, vectorizer)
ex1_result = pred  # Should be "Negative"

# --- Check ---
assert ex1_result == "Negative", f"Expected 'Negative' for a bad review, got '{ex1_result}'"
print(f"Exercise 1 passed! ✓  (Confidence: {conf:.1%})")

# --- Exercise 2: Most Positive Word ---
# Find the word with the highest positive weight in the model.
# Use vectorizer.vocabulary (list) and model.weights (array).

# YOUR CODE HERE:
top_word_idx = None  # Index of the highest weight: use np.argmax(model.weights)
top_word = None      # The word at that index: vectorizer.vocabulary[top_word_idx]

# --- Check ---
assert top_word is not None, "Find the top positive word!"
assert top_word_idx == np.argmax(model.weights), "Use np.argmax on model.weights"
print(f"Exercise 2 passed! ✓  (Most positive word: '{top_word}', weight: {model.weights[top_word_idx]:.3f})")

# --- Quick Check: BoW Limitation ---
# What is the MAIN limitation of bag-of-words for sentiment analysis?
# a) It can't handle long reviews
# b) It requires too much memory
# c) It can't handle unknown words
# d) It ignores word order ("not good" = "good not")

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'd', "Think about 'not good' vs 'good not' — BoW gives identical vectors for both!"
print("Exercise 3 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

You built a complete sentiment classifier:
- **Text preprocessing**: Clean and tokenize text
- **Bag of Words**: Convert text to numerical vectors
- **Logistic Regression**: Train a classifier
- **Interpretation**: Understand what the model learned

**Next up**: Using professional ML tools with scikit-learn! →